# 05 — Model Evaluation
**E-Waste Toxic Gas Detection System — ML Pipeline**

**Purpose:** Comprehensive evaluation of all 4 models — accuracy, precision, recall, F1, confusion matrices, ROC curves, and the final comparison table for the research paper.

**Author:** Sanjula Madushanka | Final Year Research Y4S2

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import joblib
import time
import warnings
from pathlib import Path
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report,
                             roc_auc_score, roc_curve)
from sklearn.preprocessing import label_binarize
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
DATA_DIR  = Path('../datasets/processed/train_test_split')
MODEL_DIR = Path('../models')
SAVE_DIR  = Path('../results')
CM_DIR    = SAVE_DIR / 'confusion_matrices'
CM_DIR.mkdir(parents=True, exist_ok=True)

X_test  = pd.read_csv(DATA_DIR / 'X_test.csv').values
y_test  = pd.read_csv(DATA_DIR / 'y_test.csv').values.ravel()
le      = joblib.load(MODEL_DIR / 'label_encoder.pkl')
CLASS_NAMES = list(le.classes_)

MODEL_FILES = {
    'Random Forest': 'random_forest_v1.pkl',
    'SVM':           'svm_v1.pkl',
    'Decision Tree': 'decision_tree_v1.pkl',
    'Naive Bayes':   'naive_bayes_v1.pkl',
}

models = {name: joblib.load(MODEL_DIR / fname) for name, fname in MODEL_FILES.items()}
print(f'Loaded {len(models)} models')
print(f'Classes: {CLASS_NAMES}')

## 5.1 Compute All Metrics

In [ ]:
eval_results = {}

for name, model in models.items():
    # Predict
    t0   = time.perf_counter()
    y_pred = model.predict(X_test)
    t1   = time.perf_counter()
    inference_time_ms = (t1 - t0) / len(X_test) * 1000  # ms per sample
    
    # Probabilities (for AUC)
    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X_test)
    else:
        y_prob = None
    
    # Metrics
    acc       = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    recall    = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1_w      = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    f1_macro  = f1_score(y_test, y_pred, average='macro', zero_division=0)
    cm        = confusion_matrix(y_test, y_pred)
    
    # ROC-AUC (one-vs-rest)
    if y_prob is not None and len(np.unique(y_test)) > 1:
        y_test_bin = label_binarize(y_test, classes=np.unique(y_test))
        try:
            auc = roc_auc_score(y_test_bin, y_prob, multi_class='ovr', average='weighted')
        except Exception:
            auc = None
    else:
        auc = None
    
    eval_results[name] = {
        'y_pred':          y_pred,
        'y_prob':          y_prob,
        'accuracy':        acc,
        'precision':       precision,
        'recall':          recall,
        'f1_weighted':     f1_w,
        'f1_macro':        f1_macro,
        'roc_auc':         auc,
        'confusion_matrix': cm,
        'inference_ms':    inference_time_ms,
    }

print('✅ All metrics computed')

## 5.2 Model Comparison Table (Paper-Ready)

In [ ]:
rows = []
for name, r in eval_results.items():
    rows.append({
        'Model':           name,
        'Accuracy (%)':    round(r['accuracy']    * 100, 2),
        'Precision (%)':   round(r['precision']   * 100, 2),
        'Recall (%)':      round(r['recall']      * 100, 2),
        'F1 Weighted (%)': round(r['f1_weighted'] * 100, 2),
        'F1 Macro (%)':    round(r['f1_macro']    * 100, 2),
        'ROC-AUC':         round(r['roc_auc'], 4) if r['roc_auc'] else 'N/A',
        'Infer. Time (ms)': round(r['inference_ms'], 4)
    })

comparison_df = pd.DataFrame(rows)
comparison_df = comparison_df.sort_values('F1 Weighted (%)', ascending=False).reset_index(drop=True)

# Save to CSV
comparison_df.to_csv(SAVE_DIR / 'model_comparison_table.csv', index=False)

print('=== MODEL COMPARISON TABLE (Test Set) ===')
print(comparison_df.to_string(index=False))
print(f'\n✅ Saved to {SAVE_DIR}/model_comparison_table.csv')
print('   → Use this table in your research paper Results section')

## 5.3 Confusion Matrices (All 4 Models)

In [ ]:
def plot_confusion_matrix(cm, class_names, model_name, save_path):
    cm_pct = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Raw counts
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                ax=axes[0], linewidths=0.5,
                annot_kws={'size': 12, 'weight': 'bold'})
    axes[0].set_title(f'{model_name}\nConfusion Matrix (Counts)', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Predicted Label', fontsize=11)
    axes[0].set_ylabel('True Label', fontsize=11)
    axes[0].tick_params(axis='x', rotation=45)
    
    # Percentages
    sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='RdYlGn',
                xticklabels=class_names, yticklabels=class_names,
                ax=axes[1], linewidths=0.5, vmin=0, vmax=100,
                annot_kws={'size': 11})
    axes[1].set_title(f'{model_name}\nConfusion Matrix (Row %)', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Predicted Label', fontsize=11)
    axes[1].set_ylabel('True Label', fontsize=11)
    axes[1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()


file_map = {
    'Random Forest': 'rf_confusion.png',
    'SVM':           'svm_confusion.png',
    'Decision Tree': 'dt_confusion.png',
    'Naive Bayes':   'nb_confusion.png',
}

for name, r in eval_results.items():
    plot_confusion_matrix(
        r['confusion_matrix'],
        CLASS_NAMES,
        name,
        CM_DIR / file_map[name]
    )
    print(f'  ✅ {file_map[name]}')

## 5.4 Full Classification Report

In [ ]:
for name, r in eval_results.items():
    print(f'\n{"="*55}')
    print(f'  {name} — Classification Report')
    print(f'{"="*55}')
    print(classification_report(y_test, r['y_pred'],
                                target_names=CLASS_NAMES,
                                zero_division=0))

## 5.5 Multi-Metric Comparison Bar Chart (Paper Figure)

In [ ]:
metrics = ['Accuracy (%)', 'Precision (%)', 'Recall (%)', 'F1 Weighted (%)']
model_names = comparison_df['Model'].tolist()
x = np.arange(len(model_names))
width = 0.2
bar_colors = ['#3b82f6', '#22c55e', '#f59e0b', '#ef4444']

fig, ax = plt.subplots(figsize=(13, 6))

for i, (metric, color) in enumerate(zip(metrics, bar_colors)):
    vals = comparison_df[metric].tolist()
    bars = ax.bar(x + i * width, vals, width, label=metric,
                  color=color, alpha=0.85, edgecolor='white')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{v:.1f}', ha='center', va='bottom', fontsize=8, rotation=90)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(model_names, fontsize=11)
ax.set_ylim(0, 115)
ax.set_ylabel('Score (%)', fontsize=11)
ax.set_title('Figure 10: Model Performance Comparison (Test Set)',
             fontsize=13, fontweight='bold')
ax.legend(loc='upper right', bbox_to_anchor=(1, 1))
ax.axhline(90, color='gray', linestyle='--', linewidth=1, alpha=0.6)
plt.tight_layout()
plt.savefig(SAVE_DIR / 'fig10_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved ✅')

In [ ]:
best = comparison_df.iloc[0]
print('=' * 60)
print('EVALUATION SUMMARY')
print('=' * 60)
print(f'Best model:        {best["Model"]}')
print(f'Test Accuracy:     {best["Accuracy (%)"]:.2f}%')
print(f'F1 Weighted:       {best["F1 Weighted (%)"]:.2f}%')
print(f'Inference Time:    {best["Infer. Time (ms)"]:.4f} ms/sample')
print()
print('All figures saved to results/confusion_matrices/')
print('Comparison table saved to results/model_comparison_table.csv')
print()
print('✅ Notebook 05 complete — proceed to 06_hyperparameter_tuning.ipynb')